# 10 — Statistical Significance Analysis

Rigorous evaluation of HPA, DQN, and PPO over **30 independent seeds**
(30 distinct random workloads). Because all three agents are evaluated on the
**same** seeds, the comparison is *paired*, enabling paired t-tests.

Produces:
- mean ± std per agent (the canonical report numbers)
- paired t-tests with p-values (statistical significance)
- percentage improvements with confidence

Uses the refactored modules: `env.py`, `agent.py`, `evaluate.py`.

## 1. Imports and load trained models

In [1]:
import json
import numpy as np
import torch
from scipy import stats as scipy_stats

from env import CloudClusterEnv, STEPS_PER_WEEK
from agent import ActorCritic, DQN
from evaluate import run_hpa, run_ppo, run_dqn

stats = json.load(open('trace_params.json'))['stats']

# load the freshly-trained canonical models
ppo_net = ActorCritic()
ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth'))
ppo_net.eval()

dqn_net = DQN()
dqn_net.load_state_dict(torch.load('dqn_baseline.pth'))
dqn_net.eval()

print("Models loaded. Ready for 30-seed evaluation.")

Setup complete. Steps per week: 672
CloudClusterEnv defined.
Models loaded. Ready for 30-seed evaluation.


## 2. Run all three agents over 30 paired seeds

Each agent is evaluated on the identical 30 workloads. Evaluation is fast
(no training) — expect ~1-2 minutes total.

In [2]:
import time

N_SEEDS = 30
results = {'HPA': [], 'DQN': [], 'PPO': []}
t0 = time.time()

print(f"Running {N_SEEDS} paired seeds per agent...\n")
for i in range(N_SEEDS):
    seed = 1000 + i
    results['HPA'].append(run_hpa(CloudClusterEnv(stats, seed=seed)))
    results['DQN'].append(run_dqn(CloudClusterEnv(stats, seed=seed), dqn_net))
    results['PPO'].append(run_ppo(CloudClusterEnv(stats, seed=seed), ppo_net))
    if (i+1) % 5 == 0:
        print(f"  completed {i+1}/{N_SEEDS} seeds  ({time.time()-t0:.0f}s)")

json.dump(results, open('significance_results.json', 'w'), indent=2)
print(f"\nDone in {time.time()-t0:.0f}s. Saved significance_results.json")

Running 30 paired seeds per agent...

  completed 5/30 seeds  (13s)
  completed 10/30 seeds  (27s)
  completed 15/30 seeds  (40s)
  completed 20/30 seeds  (54s)
  completed 25/30 seeds  (68s)
  completed 30/30 seeds  (81s)

Done in 81s. Saved significance_results.json


## 3. Canonical numbers — mean ± std per agent

In [3]:
print(f"{'agent':<6}{'cost (mean±std)':>22}{'breaches (mean±std)':>24}{'util':>10}")
print("-"*62)
for agent in ['HPA', 'DQN', 'PPO']:
    c = [r['cost'] for r in results[agent]]
    b = [r['breaches'] for r in results[agent]]
    u = [r['util'] for r in results[agent]]
    print(f"{agent:<6}{np.mean(c):>10.1f} ± {np.std(c):<8.1f}"
          f"{np.mean(b):>12.0f} ± {np.std(b):<8.0f}{np.mean(u):>10.3f}")

agent        cost (mean±std)     breaches (mean±std)      util
--------------------------------------------------------------
HPA        291.1 ± 2.3             1785 ± 205          0.560
DQN        169.7 ± 1.0             3684 ± 645          0.969
PPO        177.7 ± 0.7              878 ± 312          0.949


## 4. Statistical significance — paired t-tests

In [4]:
def compare(a_name, b_name, metric):
    a = np.array([r[metric] for r in results[a_name]])
    b = np.array([r[metric] for r in results[b_name]])
    t_stat, p_value = scipy_stats.ttest_rel(a, b)   # paired
    return a.mean(), b.mean(), t_stat, p_value

print("="*70)
print("PAIRED t-TESTS (n=30 seeds)")
print("="*70)
for metric in ['cost', 'breaches']:
    print(f"\n--- {metric.upper()} ---")
    for a, b in [('PPO','HPA'), ('PPO','DQN'), ('DQN','HPA')]:
        ma, mb, t, p = compare(a, b, metric)
        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
        better = a if ma < mb else b       # lower is better for both
        print(f"  {a} ({ma:.0f}) vs {b} ({mb:.0f}): t={t:.2f}, p={p:.2e} {sig}  -> {better} better")

print("\n" + "="*70)
print("*** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant")

PAIRED t-TESTS (n=30 seeds)

--- COST ---
  PPO (178) vs HPA (291): t=-290.82, p=8.41e-52 ***  -> PPO better
  PPO (178) vs DQN (170): t=38.27, p=2.24e-26 ***  -> DQN better
  DQN (170) vs HPA (291): t=-255.45, p=3.61e-50 ***  -> DQN better

--- BREACHES ---
  PPO (878) vs HPA (1785): t=-15.89, p=7.52e-16 ***  -> PPO better
  PPO (878) vs DQN (3684): t=-21.80, p=1.58e-19 ***  -> PPO better
  DQN (3684) vs HPA (1785): t=15.34, p=1.88e-15 ***  -> HPA better

*** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant


## 5. PPO improvement over HPA — with confidence

In [5]:
print("PPO improvement over HPA (per-seed paired, n=30):\n")
for metric in ['cost', 'breaches']:
    ppo_vals = np.array([r[metric] for r in results['PPO']])
    hpa_vals = np.array([r[metric] for r in results['HPA']])
    pct = (hpa_vals - ppo_vals) / hpa_vals * 100
    print(f"  {metric:<10}: {pct.mean():>5.1f}% ± {pct.std():.1f}% reduction")

# save a compact summary for the report / figures
summary = {}
for agent in ['HPA','DQN','PPO']:
    summary[agent] = {
        m: {'mean': float(np.mean([r[m] for r in results[agent]])),
            'std':  float(np.std([r[m] for r in results[agent]]))}
        for m in ['cost','breaches','util','vms']
    }
json.dump(summary, open('significance_summary.json', 'w'), indent=2)
print("\nSaved significance_summary.json")

PPO improvement over HPA (per-seed paired, n=30):

  cost      :  38.9% ± 0.4% reduction
  breaches  :  50.9% ± 15.8% reduction

Saved significance_summary.json


In [6]:
import time, os, json
import numpy as np
import torch
from env import CloudClusterEnv
from agent import ActorCritic, DQN

stats = json.load(open('trace_params.json'))['stats']

ppo_net = ActorCritic(); ppo_net.load_state_dict(torch.load('ppo_sla-focused.pth')); ppo_net.eval()
dqn_net = DQN(); dqn_net.load_state_dict(torch.load('dqn_baseline.pth')); dqn_net.eval()

env = CloudClusterEnv(stats, seed=1)
obs, _ = env.reset()
obs_t = torch.tensor(obs, dtype=torch.float32)

# ---- inference latency ----
def hpa_decision(cpu):
    return 0.2 if cpu > 0.70 else (-0.2 if cpu < 0.30 else 0.0)

n = 1000
# warmup
for _ in range(10):
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))

t0 = time.perf_counter()
for _ in range(n): hpa_decision(obs_t[0].item())
hpa_lat = (time.perf_counter()-t0)/n*1000

t0 = time.perf_counter()
for _ in range(n):
    with torch.no_grad(): ppo_net.forward(obs_t.unsqueeze(0))
ppo_lat = (time.perf_counter()-t0)/n*1000

t0 = time.perf_counter()
for _ in range(n):
    with torch.no_grad(): dqn_net(obs_t.unsqueeze(0))
dqn_lat = (time.perf_counter()-t0)/n*1000

# ---- resource overhead ----
ppo_params = sum(p.numel() for p in ppo_net.parameters())
dqn_params = sum(p.numel() for p in dqn_net.parameters())
ppo_size = os.path.getsize('ppo_sla-focused.pth')/1024
dqn_size = os.path.getsize('dqn_baseline.pth')/1024

# ---- scalability: decision time vs cluster size ----
sizes = [10, 50, 100, 500, 1000, 5000]
scal = []
for s in sizes:
    d = torch.rand(32)
    t0 = time.perf_counter()
    for _ in range(500):
        with torch.no_grad(): ppo_net.forward(d.unsqueeze(0))
    scal.append({'size': s, 'latency_ms': (time.perf_counter()-t0)/500*1000})

metrics = {
    'latency_ms': {'HPA': hpa_lat, 'DQN': dqn_lat, 'PPO': ppo_lat},
    'params': {'HPA': 0, 'DQN': dqn_params, 'PPO': ppo_params},
    'model_kb': {'HPA': 0, 'DQN': dqn_size, 'PPO': ppo_size},
    'scalability': scal,
}
json.dump(metrics, open('system_metrics.json', 'w'), indent=2)
print("Saved system_metrics.json")
print(f"Latency (ms): HPA {hpa_lat:.4f}, DQN {dqn_lat:.4f}, PPO {ppo_lat:.4f}")
print(f"Params: DQN {dqn_params:,}, PPO {ppo_params:,}")

Saved system_metrics.json
Latency (ms): HPA 0.0012, DQN 0.0174, PPO 0.0259
Params: DQN 75,525, PPO 74,755
